# 📚 Book Recommendation System
### Machine Learning Mini Project: Content-Based Filtering with TF-IDF & Cosine Similarity

This notebook demonstrates the complete Machine Learning workflow for building a content-based book recommendation system.

## 1. Import Libraries
We begin by importing the required libraries:
- **Pandas** for data manipulation and analysis
- **NumPy** for numerical computations
- **Scikit-learn** for TF-IDF vectorization and Cosine Similarity
- **Joblib** for persisting the model artifact

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries successfully imported!")

## 2. Load Dataset
Load the curated dataset stored at `data/books.csv`.

In [2]:
csv_path = os.path.join("..", "data", "books.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("data", "books.csv")

df = pd.read_csv(csv_path)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")

## 3. Dataset Overview
Inspect the dataset structure: first 5 rows, shape, column data types, and check for missing values.

In [3]:
# Display first 5 rows
df.head()

In [4]:
# Summary information and missing values
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing values count:\n", df.isnull().sum())

## 4. Data Preprocessing
Perform standard preprocessing: handle missing values, eliminate duplicates, and prepare clean text.

In [5]:
# Fill any missing values and ensure title uniqueness
df.fillna("", inplace=True)
df.drop_duplicates(subset=["title"], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Total cleaned books: {len(df)}")

## 5. Feature Engineering
In Content-Based Filtering, we combine descriptive attributes (`title`, `author`, `genre`, and `description`) into a single text representation for each book.

In [6]:
df["combined_features"] = df["title"] + " " + df["author"] + " " + df["genre"] + " " + df["description"]
df[["title", "genre", "combined_features"]].head()

## 6. TF-IDF Vectorization
**TF-IDF (Term Frequency - Inverse Document Frequency)** translates text into numerical vectors:
- **TF (Term Frequency)**: Measures how often a word appears in a specific book's metadata.
- **IDF (Inverse Document Frequency)**: Scales down words that appear across all books and boosts rare, distinctive terms (e.g., 'hobbit', 'dystopian', 'detective').

In [7]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["combined_features"])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Books: {tfidf_matrix.shape[0]}, Unique Feature Words: {tfidf_matrix.shape[1]}")

## 7. Cosine Similarity
Cosine Similarity computes the angle between vector representations of books:
$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}
The metric scores range from **0** (no topical overlap) to **1** (identical topical profile).

In [8]:
similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Similarity Matrix Shape: {similarity_matrix.shape}")

## 8. Recommendation Function
We define a straightforward function `recommend_books(book_title, n=5)` that:
1. Locates the selected book's index.
2. Retrieves its similarity scores with all other books.
3. Sorts scores descending.
4. Excludes the selected book itself.
5. Returns the top-N most similar books.

In [9]:
def recommend_books(book_title, n=5):
    if book_title not in df["title"].values:
        return f"Book '{book_title}' not found."
    
    # 1. Selected book index
    idx = df[df["title"] == book_title].index[0]
    
    # 2. Similarity scores list
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # 3. Sort descending
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # 4. Exclude selected book itself
    sim_scores = [item for item in sim_scores if item[0] != idx][:n]
    
    # 5. Extract top recommendations
    indices = [item[0] for item in sim_scores]
    scores = [round(item[1], 4) for item in sim_scores]
    
    recommendations = df.iloc[indices][["title", "author", "genre", "rating", "year"]].copy()
    recommendations["similarity_score"] = scores
    return recommendations

print("Recommendation function defined successfully!")

## 9. Test Recommendations
Let us test the recommendation system with a few diverse books.

In [10]:
print("=== Recommendations for: The Hobbit (Fantasy) ===")
print(recommend_books("The Hobbit", n=5))

print("\n=== Recommendations for: 1984 (Science Fiction) ===")
print(recommend_books("1984", n=5))

print("\n=== Recommendations for: Atomic Habits (Self-Help) ===")
print(recommend_books("Atomic Habits", n=5))

## 10. Save Model
Persist the dataset, vectorizer, and similarity matrix for seamless consumption by the Streamlit application.

In [11]:
model_dir = os.path.join("..", "model")
if not os.path.exists(model_dir):
    model_dir = "model"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "recommendation_model.pkl")
model_payload = {
    "df": df,
    "tfidf": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "similarity_matrix": similarity_matrix
}
joblib.dump(model_payload, model_path)
print(f"Model artifact successfully saved to: {model_path}")